# 📋 DAT09 · 미션4 중복과 과거 보고 및 충돌 처리하기 (학습자용)

---

## 실습 시간·작업 폴더·연결 파일

- **예상 실습 시간**: 3시간 (전체 10개 미션 총 27시간 과정)
- **작업 폴더**: `DAT09_practice`
- **연결 파일**: 첫 셀에서 `DAT09.md`를 연결하세요.

## 이전 미션 연계와 실행 결과

- **이전 미션 연계**: 같은 작업 폴더에서 미션1~미션3을 먼저 실행했다면, 첫 셀이 `src/m01.py`~`src/m03.py`를 불러와 작업 사전 `c`를 자동으로 복원합니다.

- 각 단계 함수(`m04_s1`~`m04_s5`)의 `TODO`를 직접 작성한 뒤 셀을 실행하세요.
- 실행 결과는 `outputs/M04/DAT09_M04_result.html`과 `step_01.csv`~`step_05.csv`에 저장됩니다.



In [ ]:
from pathlib import Path, PurePosixPath
import sys, io, json, re, zipfile, base64, hashlib, importlib
from IPython.display import display, HTML
import html
import pandas as pd
ROLE = 'student'
EXPECTED_SHA256 = 'a0adfd7b9a2416cd74af2fc99bb7978ad662eec8d9e40ce5b8a539977c2cb17e'
c = None

def connect_md(path):
    global c, ROOT, HERE
    path = Path(path)
    if path.name not in ('DAT09.md', 'DAT09.md'):
        raise ValueError('원자료 DAT09.md 또는 DAT09.md를 선택하세요. 정답코드 MD는 연결 파일이 아닙니다.')
    text = path.read_text(encoding='utf-8-sig')
    found = re.search(r'<!-- DAT09_RUNTIME_BUNDLE_B64:([A-Za-z0-9+/=]+) -->', text)
    if not found:
        raise ValueError('이 노트북용 원자료 블록이 없습니다. 함께 받은 DAT09.md를 사용하세요.')
    raw = base64.b64decode(found.group(1), validate=True)
    if hashlib.sha256(raw).hexdigest() != EXPECTED_SHA256:
        raise ValueError('자료와 노트북 버전이 다릅니다. 같은 묶음의 파일을 사용하세요.')
    ROOT = Path.cwd() / ('DAT09_reference' if ROLE == 'solution' else 'DAT09_practice')
    ROOT.mkdir(exist_ok=True)
    with zipfile.ZipFile(io.BytesIO(raw)) as archive:
        for info in archive.infolist():
            name = PurePosixPath(info.filename)
            if name.is_absolute() or '..' in name.parts:
                raise ValueError('허용되지 않은 경로입니다.')
            target = ROOT.joinpath(*name.parts)
            target.parent.mkdir(parents=True, exist_ok=True)
            target.write_bytes(archive.read(info))
    sys.path.insert(0, str(ROOT))
    for name in list(sys.modules):
        if name in ['dat09_common', 'dat09_ui'] or name.startswith('src.m'):
            del sys.modules[name]
    import dat09_common
    globals().update({name: getattr(dat09_common, name) for name in dir(dat09_common) if not name.startswith('_')})
    import collections, itertools, queue, threading, tempfile, importlib.util, shutil
    globals().update({k:v for k,v in locals().items() if k in ['collections','itertools','re','queue','threading','tempfile','importlib','shutil']})
    from dat09_ui import bars, heatmap, line_chart, dashboard_fragment
    globals().update(bars=bars,heatmap=heatmap,line_chart=line_chart,dashboard_fragment=dashboard_fragment)
    c = dat09_common.context(ROOT)
    HERE = ROOT.parent / 'instructor'
    for candidate in [Path.cwd()/'instructor', Path.cwd(), ROOT.parent/'instructor']:
        if (candidate/'aar_key.json').is_file():
            HERE = candidate
            c['instructor_root'] = candidate
            break
    print('자료 연결 완료:', ROOT)
    print('10개 미션·50개 함수·27시간 과정입니다. 미션1부터 실행하세요.')
    print('TODO가 남아 있으면 해당 단계에서 멈춥니다. src에는 작성한 함수를 저장합니다.')
    return c

def hydrate_previous_missions(c, up_to=3):
    """같은 작업 폴더에 저장된 이전 미션(src/m01.py~)을 순차 실행해 작업 사전 c를 복원합니다."""
    import importlib.util
    src_dir = c['root'] / 'src'
    missing = [k for k in range(1, up_to + 1) if not (src_dir / f'm{k:02d}.py').is_file()]
    if missing:
        miss_str = ', '.join(f'미션{k}(DAT09_m{k:02d}_*.ipynb)' for k in missing)
        print(f'[안내] 아직 완료되지 않은 이전 미션이 있습니다: {miss_str}')
        print('이전 미션 노트북을 같은 폴더에서 먼저 실행한 뒤 이 셀을 다시 실행하세요.')
        return False
    for k in range(1, up_to + 1):
        if k in c['completed']:
            continue
        mod_path = src_dir / f'm{k:02d}.py'
        spec = importlib.util.spec_from_file_location(f'dat09_prev_m{k:02d}', mod_path)
        mod = importlib.util.module_from_spec(spec)
        spec.loader.exec_module(mod)
        fns = [getattr(mod, f'm{k:02d}_s{s}') for s in range(1, 6)]
        run_mission(c, k, fns)
    print(f'이전 미션(미션1~미션{up_to}) 상태 복원 완료.')
    return True

DATA_MD = next((Path(p) for p in ('DAT09.md', 'DAT09.md') if Path(p).is_file()), None)
if DATA_MD is not None:
    connect_md(DATA_MD)
    hydrate_previous_missions(c, 3)
else:
    print('DAT09.md(또는 DAT09.md)를 이 노트북과 같은 폴더에 두고 첫 셀을 다시 실행하세요.')
    print("다른 경로에 있다면 connect_md('/자료가_있는_경로/DAT09.md'); hydrate_previous_missions(c)를 실행하세요.")


## 미션4 중복과 과거 보고 및 충돌 처리하기

예상 3시간입니다.

#### 이번 미션의 목표

보고가 중복되거나 순서가 바뀌어도 현재값을 지키고, 원인을 확인한 범위 안에서만 보완 후보를 제안합니다.

### 1단계 이벤트의 수량과 버전 및 시각 뜻 정하기

#### 이 단계에서 하는 일

이벤트는 새로 도착한 보고 한 건입니다. 이번 수량은 늘거나 줄어든 양이 아니라 보고 시점의 총보유량입니다. 이를 증감량처럼 계속 더하면 재전송 때마다 재고가 늘어납니다.

#### 시작 전 확인

미션3까지 완료한 결과를 사용합니다. 노트북의 m04_s1(c)를 작성하세요. c는 같은 작업의 자료와 함수를 담는 사전입니다. 필요한 결과를 c에 보관하고 화면에 보여 줄 표를 반환합니다.

#### 사용할 자료

data/events.jsonl의 110건과 data/event_initial_state.json의 90개 현재 상태를 사용합니다. JSONL은 한 줄마다 JSON 보고 하나를 저장한 형식입니다.

#### 작업 순서

- 파일의 줄마다 이벤트를 읽고 event_id, source, 품목·거점, metric, version, quantity, unit, as_of, observed_at, received_at을 확인하세요.

- 현재 상태를 찾는 조합은 출처·거점·품목·지표로 정합니다. 수량은 그 시점의 총수량이며 version이 클수록 새 보고라는 규칙을 문서로 남기세요.

- as_of는 보고 기준시각, observed_at은 재고를 실제 관측한 시각, received_at은 받은 시각으로 구별하세요. 파일에서 늦게 읽었다는 이유로 더 최신이라고 판단하지 않습니다.

- 이벤트 목록과 초기 상태를 c['events'], c['initial']에 저장하세요. 외부 메시지 서버 대신 Python 자료구조와 메모리 큐로 실습합니다.


#### 저장할 결과

이벤트 항목 설명과 입력·초기 상태 건수 표를 남깁니다. 뒤 단계의 처리 결과와 섞지 않습니다.

#### 직접 확인하는 방법

입력 110건과 초기 상태 90개를 확인하세요. 현재 80EA 뒤에 버전이 높은 90EA 보고를 받으면 현재값은 170이 아니라 90EA가 됩니다.

#### 실행과 기록

이 미션의 다섯 함수를 작성한 뒤 코드셀을 실행합니다. 반환 표는 outputs/M04/step_01.csv에 저장됩니다. 전체 결과는 같은 폴더의 DAT09_M04_result.html에서 확인합니다. 사람이 확인한 내용은 review_notes.md에 단계 번호·사례·근거·판단을 직접 적으세요. 앞 단계 자료가 없으면 이전 미션부터 순서대로 실행합니다.

### 2단계 중복과 과거 보고 및 충돌을 구분해 처리하기

#### 이 단계에서 하는 일

같은 보고의 재전송은 한 번만 반영하고, 과거 보고는 새 값을 덮어쓰지 못하게 해야 합니다. 같은 버전인데 내용이 다르면 어느 쪽이 맞는지 코드만으로 정할 수 없으므로 충돌로 남깁니다.

#### 시작 전 확인

미션4의 1단계까지 작성·확인한 결과를 사용합니다. 노트북의 m04_s2(c)를 작성하세요. c는 같은 작업의 자료와 함수를 담는 사전입니다. 필요한 결과를 c에 보관하고 화면에 보여 줄 표를 반환합니다.

#### 사용할 자료

앞 단계의 이벤트·초기 상태, 미션2 코드·단위·시각 기준을 사용합니다. 출처와 event_id를 함께 묶어 이미 받은 보고인지 확인합니다.

#### 작업 순서

- 이벤트를 등록된 코드·EA·UTC로 바꾸고 유효성을 확인하는 SyncEngine을 만드세요. 처음 보는 높은 버전은 APPLIED, 같은 ID의 같은 내용은 DUPLICATE로 기록합니다.

- 낮은 버전은 STALE로 기록하고 현재값을 유지하세요. 같은 버전의 서로 다른 수량은 CONFLICT로 두고, 이전 정상값과 충돌한 후보들을 함께 보관합니다.

- 같은 ID인데 내용이 달라진 경우도 충돌로 기록하세요. 한쪽을 임의로 채택하거나 단순 중복으로 버리지 않습니다.

- 받은 순서·이벤트 ID·대상·상태·이유·처리시간을 event_ledger.csv에 남기세요. 중간 상태를 체크포인트 파일로 저장합니다. 체크포인트는 중단 후 이어서 처리하기 위한 저장본입니다.


#### 저장할 결과

event_ledger.csv와 checkpoint.json을 저장하고 c['SyncEngine'], c['engine']에 처리기와 실행 상태를 남깁니다.

#### 직접 확인하는 방법

제공된 원순서에서는 110건 중 적용 90·중복 12·과거 6·충돌 보고 2건입니다. 최종 충돌 대상은 2개이고 전체 현재 상태는 90개입니다.

#### 실행과 기록

이 미션의 다섯 함수를 작성한 뒤 코드셀을 실행합니다. 반환 표는 outputs/M04/step_02.csv에 저장됩니다. 전체 결과는 같은 폴더의 DAT09_M04_result.html에서 확인합니다. 사람이 확인한 내용은 review_notes.md에 단계 번호·사례·근거·판단을 직접 적으세요. 앞 단계 자료가 없으면 이전 미션부터 순서대로 실행합니다.

### 3단계 순서 변경과 중단 후 재개의 최종 결과 대조하기

#### 이 단계에서 하는 일

합계만 같아도 품목 두 개의 값이 서로 바뀌어 있을 수 있습니다. 처리 순서가 달라도 대상별 최종값과 충돌 대상이 같아야 합니다. 검산용 결과는 처리기의 같은 함수를 다시 부르는 방식과 분리합니다.

#### 시작 전 확인

미션4의 2단계까지 작성·확인한 결과를 사용합니다. 노트북의 m04_s3(c)를 작성하세요. c는 같은 작업의 자료와 함수를 담는 사전입니다. 필요한 결과를 c에 보관하고 화면에 보여 줄 표를 반환합니다.

#### 사용할 자료

events.jsonl, 초기 상태와 앞 단계의 SyncEngine·체크포인트를 사용합니다. 섞는 순서는 재현할 수 있게 난수 시드를 고정합니다.

#### 작업 순서

- 전체 이벤트를 한꺼번에 모아 ID 중복을 제거하고, 대상·버전별로 묶어 최고 유효 버전을 고르는 검산 코드를 별도로 만드세요. 최고 버전이 충돌하면 이전 정상값을 유지합니다.

- 실시간 처리기는 원순서, 역순, 시드를 고정해 섞은 순서로 각각 새 초기 상태에서 실행하세요.

- 원순서의 절반쯤에서 체크포인트를 저장하고 처리기를 새로 만든 뒤 복원해 나머지를 처리하세요. 이미 본 ID와 충돌 정보도 복원해야 합니다.

- 네 방식 각각의 대상별 버전·수량·기준시각과 충돌 대상 목록을 일괄 검산 결과와 비교하세요. 같은 ID의 다른 내용이 충돌로 남는 추가 시험도 실행합니다.


#### 저장할 결과

batch_stream_diff.csv와 restart_checkpoint.json, 별도 검산 코드 및 비교 결과를 저장합니다.

#### 직접 확인하는 방법

원순서·역순·섞기·재개 네 방식에서 대상별 차이 0개와 충돌 목록 차이 없음이 나와야 합니다. 총수량만 같다는 확인으로 끝내지 않습니다.

#### 실행과 기록

이 미션의 다섯 함수를 작성한 뒤 코드셀을 실행합니다. 반환 표는 outputs/M04/step_03.csv에 저장됩니다. 전체 결과는 같은 폴더의 DAT09_M04_result.html에서 확인합니다. 사람이 확인한 내용은 review_notes.md에 단계 번호·사례·근거·판단을 직접 적으세요. 앞 단계 자료가 없으면 이전 미션부터 순서대로 실행합니다.

### 4단계 원인 분류와 보완 후보의 근거 남기기

#### 이 단계에서 하는 일

수량 차이만으로 입력 실수라고 단정할 수 없습니다. 단위가 다른지, 보고가 오래됐는지, 재계수로 오류를 확인했는지를 구별해야 합니다. 설명 가능한 근거로 분류하고 확인되지 않은 원인은 모른다고 남깁니다.

#### 시작 전 확인

미션4의 3단계까지 작성·확인한 결과를 사용합니다. 노트북의 m04_s4(c)를 작성하세요. c는 같은 작업의 자료와 함수를 담는 사전입니다. 필요한 결과를 c에 보관하고 화면에 보여 줄 표를 반환합니다.

#### 사용할 자료

data/cause_dev.json의 개발 사례, cause_test.json의 별도 평가 사례, 미션3의 출처별 관측 묶음을 사용합니다. 평가용 정답표는 규칙 작성에 사용하지 않습니다.

#### 작업 순서

- 개발 사례를 보고 결측 MISSING, 단위 차이 UNIT_MISMATCH, 지연 LAG, 확인된 입력 오류 INPUT_ERROR_CONFIRMED, 원인 미확인 UNKNOWN의 다섯 분류 규칙을 작성하세요.

- 재계수 참조번호와 재계수 수량이 있을 때만 입력 오류가 확인됐다고 판단하세요. 수량 차이만 있으면 UNKNOWN으로 남깁니다. 여러 조건이 겹칠 때 검사 순서도 기록합니다.

- 평가 사례의 case_id·예측 분류·판단 이유·재계수 후보를 먼저 cause_predictions.csv에 저장하세요. 저장 전에는 교관 정답을 읽지 않습니다.

- 실제 보고 묶음에서 최신·유효한 두 출처가 같은 수량을 제시하면 검토 후보를 별도로 만드세요. 근거 보고 ID·출처 버전·후보 버전·제안자와 미승인 상태를 남깁니다.


#### 저장할 결과

cause_predictions.csv와 candidate_reasons.csv를 저장합니다. 분류 함수·예측·후보는 다음 단계에서 사용할 수 있도록 c에 보관합니다.

#### 직접 확인하는 방법

원인 분류와 수량 후보는 서로 다른 결과입니다. 단위 차이를 발견했다고 실재고가 확인된 것은 아니며, 두 출처가 같아도 승인 없이 확정하지 않습니다.

#### 실행과 기록

이 미션의 다섯 함수를 작성한 뒤 코드셀을 실행합니다. 반환 표는 outputs/M04/step_04.csv에 저장됩니다. 전체 결과는 같은 폴더의 DAT09_M04_result.html에서 확인합니다. 사람이 확인한 내용은 review_notes.md에 단계 번호·사례·근거·판단을 직접 적으세요. 앞 단계 자료가 없으면 이전 미션부터 순서대로 실행합니다.

### 5단계 분류 정확도와 후보 제안 범위를 구분해 평가하기

#### 이 단계에서 하는 일

후보를 거의 내지 않고 맞춘 두 건만 보고 정확도가 높다고 말하면 성과를 오해할 수 있습니다. 분류가 맞은 비율, 종류별 누락, 후보를 낸 범위, 정답이 있는 후보의 정확도를 따로 계산합니다.

#### 시작 전 확인

미션4의 4단계까지 작성·확인한 결과를 사용합니다. 노트북의 m04_s5(c)를 작성하세요. c는 같은 작업의 자료와 함수를 담는 사전입니다. 필요한 결과를 c에 보관하고 화면에 보여 줄 표를 반환합니다.

#### 사용할 자료

앞 단계에서 저장한 예측을 고정한 뒤 교관이 제공하는 cause_test_key.json과 비교합니다. 정답표가 없으면 해당 지표는 미채점으로 남깁니다.

#### 작업 순서

- 실제 분류를 행, 예측 분류를 열로 둔 혼동행렬을 만드세요. 각 칸은 그 실제 분류를 해당 분류로 예측한 건수입니다. 전체 정답수/전체 평가수로 정확도를 계산합니다.

- 분류별로 TP는 맞게 찾은 건수, FP는 다른 분류를 그 분류로 잘못 고른 건수, FN은 그 분류를 놓친 건수로 셉니다. 재현율은 TP/(TP+FN), F1은 2TP/(2TP+FP+FN)이며 macro F1은 다섯 분류 F1의 단순 평균입니다.

- 후보 제안 비율은 후보를 낸 평가 건수/전체 평가 건수로 계산하세요. 후보 정확도는 정답 수량과 제안 수량이 모두 있는 건만 분모로 삼고 그중 수량이 같은 건을 셉니다.

- 분자가 없거나 분모가 0이면 계산할 수 없다고 표시하세요. 기본 예제의 높은 점수가 다른 자료에서도 유지된다고 단정하지 말고 틀린 사례나 표본 수의 한계를 적습니다.


#### 저장할 결과

confusion_matrix.csv, sync_report.json과 분류별 재현율·F1 표를 저장합니다. 정답표 미제공이면 지표값 대신 미채점 사유를 남깁니다.

#### 직접 확인하는 방법

기본 평가 사례는 10건이며 정답 구현의 분류는 10/10입니다. 재계수 후보 제안은 2/10, 정답 수량이 있는 후보 비교는 2건이 분모입니다. 서로 다른 분모를 섞지 않습니다.

#### 실행과 기록

이 미션의 다섯 함수를 작성한 뒤 코드셀을 실행합니다. 반환 표는 outputs/M04/step_05.csv에 저장됩니다. 전체 결과는 같은 폴더의 DAT09_M04_result.html에서 확인합니다. 사람이 확인한 내용은 review_notes.md에 단계 번호·사례·근거·판단을 직접 적으세요. 앞 단계 자료가 없으면 이전 미션부터 순서대로 실행합니다.



In [ ]:
if c is None:
    raise RuntimeError("첫 셀에서 DAT09.md(또는 DAT09.md)를 먼저 연결하세요.")

# STEP 01 이벤트의 수량과 버전 및 시각 뜻 정하기
# 할 일: 파일의 줄마다 이벤트를 읽고 event_id, source, 품목·거점, metric, version, quantity, unit, as_of, observed_at, received_at을 확인하세요. 현재 상태를 찾는 조합은 출처·거점·품목·지표로 정합니다. 수량은 그 시점의 총수량이며 version이 클수록 새 보고라는 규칙을 문서로 남기세요. as_of는 보고 기준시각, observed_at은 재고를 실제 관측한 시각, received_at은 받은 시각으로 구별하세요. 파일에서 늦게 읽었다는 이유로 더 최신이라고 판단하지 않습니다. 이벤트 목록과 초기 상태를 c['events'], c['initial']에 저장하세요. 외부 메시지 서버 대신 Python 자료구조와 메모리 큐로 실습합니다.
# 확인: 입력 110건과 초기 상태 90개를 확인하세요. 현재 80EA 뒤에 버전이 높은 90EA 보고를 받으면 현재값은 170이 아니라 90EA가 됩니다.
def m04_s1(c):
    """이벤트의 수량과 버전 및 시각 뜻 정하기.

    읽을 작업값: root
    저장할 작업값: events, initial
    결과 폴더: folder(c, 4)
    """
    return TODO(4, 1)

# STEP 02 중복과 과거 보고 및 충돌을 구분해 처리하기
# 할 일: 이벤트를 등록된 코드·EA·UTC로 바꾸고 유효성을 확인하는 SyncEngine을 만드세요. 처음 보는 높은 버전은 APPLIED, 같은 ID의 같은 내용은 DUPLICATE로 기록합니다. 낮은 버전은 STALE로 기록하고 현재값을 유지하세요. 같은 버전의 서로 다른 수량은 CONFLICT로 두고, 이전 정상값과 충돌한 후보들을 함께 보관합니다. 같은 ID인데 내용이 달라진 경우도 충돌로 기록하세요. 한쪽을 임의로 채택하거나 단순 중복으로 버리지 않습니다. 받은 순서·이벤트 ID·대상·상태·이유·처리시간을 event_ledger.csv에 남기세요. 중간 상태를 체크포인트 파일로 저장합니다. 체크포인트는 중단 후 이어서 처리하기 위한 저장본입니다.
# 확인: 제공된 원순서에서는 110건 중 적용 90·중복 12·과거 6·충돌 보고 2건입니다. 최종 충돌 대상은 2개이고 전체 현재 상태는 90개입니다.
def m04_s2(c):
    """중복과 과거 보고 및 충돌을 구분해 처리하기.

    읽을 작업값: catalog, checks, events, initial
    저장할 작업값: SyncEngine, engine
    결과 폴더: folder(c, 4)
    """
    return TODO(4, 2)

# STEP 03 순서 변경과 중단 후 재개의 최종 결과 대조하기
# 할 일: 전체 이벤트를 한꺼번에 모아 ID 중복을 제거하고, 대상·버전별로 묶어 최고 유효 버전을 고르는 검산 코드를 별도로 만드세요. 최고 버전이 충돌하면 이전 정상값을 유지합니다. 실시간 처리기는 원순서, 역순, 시드를 고정해 섞은 순서로 각각 새 초기 상태에서 실행하세요. 원순서의 절반쯤에서 체크포인트를 저장하고 처리기를 새로 만든 뒤 복원해 나머지를 처리하세요. 이미 본 ID와 충돌 정보도 복원해야 합니다. 네 방식 각각의 대상별 버전·수량·기준시각과 충돌 대상 목록을 일괄 검산 결과와 비교하세요. 같은 ID의 다른 내용이 충돌로 남는 추가 시험도 실행합니다.
# 확인: 원순서·역순·섞기·재개 네 방식에서 대상별 차이 0개와 충돌 목록 차이 없음이 나와야 합니다. 총수량만 같다는 확인으로 끝내지 않습니다.
def m04_s3(c):
    """순서 변경과 중단 후 재개의 최종 결과 대조하기.

    읽을 작업값: SyncEngine, catalog, checks, events, initial
    저장할 작업값: stream_replay_results
    결과 폴더: folder(c, 4)
    """
    return TODO(4, 3)

# STEP 04 원인 분류와 보완 후보의 근거 남기기
# 할 일: 개발 사례를 보고 결측 MISSING, 단위 차이 UNIT_MISMATCH, 지연 LAG, 확인된 입력 오류 INPUT_ERROR_CONFIRMED, 원인 미확인 UNKNOWN의 다섯 분류 규칙을 작성하세요. 재계수 참조번호와 재계수 수량이 있을 때만 입력 오류가 확인됐다고 판단하세요. 수량 차이만 있으면 UNKNOWN으로 남깁니다. 여러 조건이 겹칠 때 검사 순서도 기록합니다. 평가 사례의 case_id·예측 분류·판단 이유·재계수 후보를 먼저 cause_predictions.csv에 저장하세요. 저장 전에는 교관 정답을 읽지 않습니다. 실제 보고 묶음에서 최신·유효한 두 출처가 같은 수량을 제시하면 검토 후보를 별도로 만드세요. 근거 보고 ID·출처 버전·후보 버전·제안자와 미승인 상태를 남깁니다.
# 확인: 원인 분류와 수량 후보는 서로 다른 결과입니다. 단위 차이를 발견했다고 실재고가 확인된 것은 아니며, 두 출처가 같아도 승인 없이 확정하지 않습니다.
def m04_s4(c):
    """원인 분류와 보완 후보의 근거 남기기.

    읽을 작업값: groups, root
    저장할 작업값: candidates, cause_predictions, classify_fn
    결과 폴더: folder(c, 4)
    """
    return TODO(4, 4)

# STEP 05 분류 정확도와 후보 제안 범위를 구분해 평가하기
# 할 일: 실제 분류를 행, 예측 분류를 열로 둔 혼동행렬을 만드세요. 각 칸은 그 실제 분류를 해당 분류로 예측한 건수입니다. 전체 정답수/전체 평가수로 정확도를 계산합니다. 분류별로 TP는 맞게 찾은 건수, FP는 다른 분류를 그 분류로 잘못 고른 건수, FN은 그 분류를 놓친 건수로 셉니다. 재현율은 TP/(TP+FN), F1은 2TP/(2TP+FP+FN)이며 macro F1은 다섯 분류 F1의 단순 평균입니다. 후보 제안 비율은 후보를 낸 평가 건수/전체 평가 건수로 계산하세요. 후보 정확도는 정답 수량과 제안 수량이 모두 있는 건만 분모로 삼고 그중 수량이 같은 건을 셉니다. 분자가 없거나 분모가 0이면 계산할 수 없다고 표시하세요. 기본 예제의 높은 점수가 다른 자료에서도 유지된다고 단정하지 말고 틀린 사례나 표본 수의 한계를 적습니다.
# 확인: 기본 평가 사례는 10건이며 정답 구현의 분류는 10/10입니다. 재계수 후보 제안은 2/10, 정답 수량이 있는 후보 비교는 2건이 분모입니다. 서로 다른 분모를 섞지 않습니다.
def m04_s5(c):
    """분류 정확도와 후보 제안 범위를 구분해 평가하기.

    읽을 작업값: cause_predictions, checks
    저장할 작업값: 근거 파일과 반환 표
    결과 폴더: folder(c, 4)
    """
    return TODO(4, 5)

result_html = run_mission(c, 4, [m04_s1, m04_s2, m04_s3, m04_s4, m04_s5])
document = result_html.read_text(encoding="utf-8")
display(HTML('<iframe title="DAT09 단계별 결과" style="width:100%;height:1100px;border:1px solid #ccd5dd" sandbox="allow-scripts allow-downloads allow-same-origin" srcdoc="'+html.escape(document,quote=True)+'"></iframe>'))
